# 06 — Sénat : vérification de provenance (eFD)

**Rôle :** vérifier, sur un petit échantillon, que le socle ouvert (notebook 05) correspond bien aux dépôts **officiels** d'efdsearch.

**Pas d'évasion, pas d'automatisation du gate.** Ce notebook **ne scrape pas** efdsearch. Vous acceptez l'agrément **manuellement dans votre navigateur**, vous enregistrez quelques PTR (HTML ou PDF) dans `data/external/senate_efd_samples/`, et ce notebook les lit pour comparaison. C'est volontaire : on ne contourne ni Akamai ni les conditions d'usage (cf. le drapeau §105(c)).

**Cellule 1 — Chemins & dossier d'échantillons.** (`pip install lxml` pour `read_html`.)

In [1]:
from pathlib import Path
import pandas as pd

def find_project_root(start: Path) -> Path:
    for p in [start, *start.parents]:
        if (p / "requirements.txt").exists() or (p / ".git").exists():
            return p
    return start

PROJECT_ROOT = find_project_root(Path.cwd())
SAMPLES      = PROJECT_ROOT / "data" / "external" / "senate_efd_samples"; SAMPLES.mkdir(parents=True, exist_ok=True)
PROC_SENATE  = PROJECT_ROOT / "data" / "processed" / "senate"
print("Déposez vos PTR eFD (.html/.pdf) dans :", SAMPLES)

Déposez vos PTR eFD (.html/.pdf) dans : /Users/lemairealice/Downloads/Jupiter/semaine 1/data/external/senate_efd_samples


**Cellule 2 — Lister les échantillons déposés manuellement.**

In [2]:
files = sorted(list(SAMPLES.glob("*.html")) + list(SAMPLES.glob("*.pdf")))
print(f"{len(files)} échantillon(s)")
for f in files:
    print(" -", f.name)

0 échantillon(s)


**Cellule 3 — Lecture d'un PTR électronique (HTML).** Les dépôts électroniques eFD sont des tableaux propres ; `pandas.read_html` lit directement les transactions.

In [3]:
def read_efd_html(path):
    try:
        for t in pd.read_html(path):
            cols = " ".join(map(str, t.columns)).lower()
            if "ticker" in cols or "transaction" in cols:
                return t
    except Exception as e:
        print("lecture impossible:", path.name, e)
    return None

tables = {f.name: read_efd_html(f) for f in files if f.suffix == ".html"}
print("Tableaux lus :", sum(v is not None for v in tables.values()))

Tableaux lus : 0


**Cellule 4 — Recoupement avec le socle.** Comparer déclarant / ticker / date ; tout écart est **noté, jamais corrigé en silence**.

In [4]:
socle = pd.read_csv(PROC_SENATE / "senate_transactions.csv")
print("Socle :", len(socle), "lignes.")
print("Comparez ici les tickers/dates de vos échantillons eFD au socle (ex. un ticker vu doit exister pour ce déclarant).")

Socle : 12263 lignes.
Comparez ici les tickers/dates de vos échantillons eFD au socle (ex. un ticker vu doit exister pour ce déclarant).


Provenance vérifiable ✅ — passez à l'unification **`07_unify_clean`**.